> 在线讲义：[第 15 章 · 损失与优化器：cross_entropy 的细节和 AdamW 的一步](https://wikieden.github.io/pytorch-inside-out/lessons/15-loss-optimizer.html) · 本书《PyTorch 从里到外》

# 第 15 章 · 损失与优化器：cross_entropy 的细节和 AdamW 的一步

本章回答：`F.cross_entropy` 到底接收什么、算什么？语言模型的 (B, T, V) 输出怎么喂给它？`ignore_index`、`label_smoothing` 干什么？AdamW 一步更新到底怎么算，为什么梯度很小它也走一样大的步？`fused=True` 快在哪？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/15-loss-optimizer.md`（讲解、动画、练习都在讲义里）。对应原文：00-overview.md §4.6–§4.7；CS336 a1 handout §4（cross-entropy、AdamW 算法）；Loshchilov & Hutter「Decoupled Weight Decay Regularization」（AdamW，〔在线〕arXiv:1711.05101）Algorithm 2；超越 Transformer 第 1 章 交叉熵；本章数字在本机 torch 2.15.0 上跑过

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 cross_entropy | −log p 正确 | `ce_examples()['ce']` | 0.4170 |
| 第二步 ignore / 两个都算 / 平滑 | — | `ce_examples()` | 0.4170 / 0.3069 / 0.5137 |
| 第三步 AdamW | 衰减 + m̂/√v̂ | `adamw_steps()` | 0.899、0.7981、0.6973；m 0.05，v 0.00025 |
| 第三步 小梯度 | 首步 ≈ lr | `adamw_first_step_small_grad()` | 0.9000 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `adamw_first_step_small_grad`**（来自 `ch15_loss_optimizer.py`）：下面要用到的小工具。

In [ ]:
def adamw_first_step_small_grad(g=0.0005):
    p = torch.nn.Parameter(torch.tensor([1.0]))
    opt = torch.optim.AdamW([p], lr=0.1, weight_decay=0.0)
    p.grad = torch.tensor([g])
    opt.step()
    return p.item()

**函数 `adamw_steps`**（来自 `ch15_loss_optimizer.py`）：下面要用到的小工具。

In [ ]:
def adamw_steps(theta=1.0, g=0.5, lr=0.1, wd=0.01, steps=3):
    p = torch.nn.Parameter(torch.tensor([theta]))
    opt = torch.optim.AdamW([p], lr=lr, weight_decay=wd)
    out = []
    for _ in range(steps):
        p.grad = torch.tensor([g])
        opt.step()
        st = opt.state[p]
        out.append((round(p.item(), 6), st["exp_avg"].item(), st["exp_avg_sq"].item()))
    return out

**函数 `ce_examples`**（来自 `ch15_loss_optimizer.py`）：下面要用到的小工具。

In [ ]:
def ce_examples():
    lg = torch.tensor([[2.0, 1.0, 0.1]])
    lg2 = torch.tensor([[2.0, 1.0, 0.1], [0.5, 2.5, 0.0]])
    return dict(
        softmax=torch.softmax(lg, -1).tolist(),
        ce=F.cross_entropy(lg, torch.tensor([0])).item(),
        ignore=F.cross_entropy(lg2, torch.tensor([0, -100])).item(),
        both=F.cross_entropy(lg2, torch.tensor([0, 1])).item(),
        smooth=F.cross_entropy(lg, torch.tensor([0]), label_smoothing=0.1).item(),
    )

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一、二步 cross_entropy：", ce_examples())

In [ ]:
print("\n第三步 AdamW 三步 (θ, m, v)：", adamw_steps())
print("梯度 0.0005 时第一步后 θ：", adamw_first_step_small_grad())

## 3. 代码里才看得到的

1. **`-100` 不进分母**：两个样本忽略一个，平均只除以 1。
2. **`opt.state[p]['step']` 是张量**：第几步也存在状态里，存档要一起存（第 10 章）。

## 4. 自己改一改

- `adamw_steps(g=-0.5)`，看方向。
- `ce_examples` 里把 label_smoothing 改成 0.3。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> cross_entropy 吃 logits 别先 softmax，算的是负 log 正确概率；
> 语言模型拍平再算，补齐位 ignore_index；
> AdamW 先衰减，再用惯性除以平时梯度，首步约等学习率；fused 一个 kernel，bias 和 norm 不衰减。